# Validation Test：STR 量稳换手率因子严格验证（2018–2021）

这份 Notebook 只负责样本外验证，不重复计算因子：

- **预热集：2017 年末历史行情**，由因子生产 Notebook 使用；
- **训练集：2006-01-04—2017-12-29**，用于确定因子定义、方向和回测参数；
- **验证集：2018-01-02—2021-12-31**，本 Notebook 只评价，不根据结果调参；
- **测试集：2022-01-04以后**，保持锁定，本 Notebook 不读取测试期因子、不计算测试期收益。

运行顺序：冻结协议 → 读取预生成验证因子 → 生成严格可交易标记 → T+1 VWAP 回测 → 成本压力测试 → 按预设门槛给出结论。


## 1. 集中配置与冻结协议

所有需要人工调整的项目都集中在下方唯一的“用户修改区”。其余路径由这些参数自动派生，后续单元不再重复配置。

验证因子必须先由 `Factor/TURNOVER_STABILITY.ipynb` 使用相同训练定义生成。验证 Notebook 只读取因子 Parquet，避免维护第二套因子实现。


In [ ]:
from pathlib import Path
from IPython.display import display
import gc
import importlib.util
import json
import os
import sys

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:.6f}')

LEARNING_DIR = Path(os.getenv('QUANT_LEARNING_DIR', str(next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'Back_test').exists() and (p / 'Factor').exists()), Path.cwd()))))
VALIDATION_DIR = Path(os.getenv(
    'VALIDATIONTEST_WORK_DIR', str(LEARNING_DIR / 'Validation_test')
))
BACKTEST_DIR = LEARNING_DIR / 'Back_test'
BACKTEST_ENGINE = BACKTEST_DIR / 'backtest.py'

# 明确加载本地引擎，避免目录大小写错误和同名 backtest 包冲突。
assert BACKTEST_ENGINE.is_file(), f'回测引擎不存在：{BACKTEST_ENGINE}'
engine_spec = importlib.util.spec_from_file_location(
    'strict_factor_backtest_engine_validation', BACKTEST_ENGINE
)
assert engine_spec is not None and engine_spec.loader is not None
backtest_engine = importlib.util.module_from_spec(engine_spec)
sys.modules[engine_spec.name] = backtest_engine
engine_spec.loader.exec_module(backtest_engine)

BacktestConfig = backtest_engine.BacktestConfig
DataPaths = backtest_engine.DataPaths
FactorSpec = backtest_engine.FactorSpec
StrictFactorBacktester = backtest_engine.StrictFactorBacktester


# ============================================================================
# ============================ 用户唯一修改区 ================================
# ============================================================================

# 1) 样本边界：验证期和测试期不得重叠。
TRAIN_END_DATE = os.getenv('VALIDATIONTEST_TRAIN_END_DATE', '2017-12-29')
VALIDATION_START_DATE = os.getenv('VALIDATIONTEST_START_DATE', '2018-01-02')
VALIDATION_END_DATE = os.getenv('VALIDATIONTEST_END_DATE', '2021-12-31')
TEST_START_DATE = os.getenv('VALIDATIONTEST_TEST_START_DATE', '2022-01-04')

# 2) 因子身份与预生成文件命名。
FACTOR_NAME = 'volume_price_divergence'
FACTOR_COLUMN = 'volume_price_divergence_factor'
FACTOR_DIRECTION = 'positive'
FACTOR_OUTPUT_PREFIX = 'volume_price_divergence_output'
FACTOR_FILE_PREFIX = 'volume_price_divergence_factor'
FACTOR_PREPROCESSED = True

# 验证因子及覆盖率文件所在的根目录。
FACTOR_DATA_ROOT = LEARNING_DIR / 'Validation_test' / 'data'

# 3) 冻结的训练回测参数：验证主结果必须保持一致。
HORIZON = 5
EXECUTION_LAG = 1
REBALANCE_STEP = 5
GROUPS = 10
BASE_COST_BPS = 10.0
MIN_STOCKS = 100
MAX_MISSING_RETURN_RATIO = 0.40
MARKET_EXCHANGES = ('SH', 'SZ')
SAVE_HOLDINGS = True

# 4) 更严格的验证设置。
PRICE_LIMIT_TOLERANCE = 0.005
COST_STRESS_BPS = (10.0, 20.0, 30.0, 50.0)
STRICT_COST_GATE_BPS = 20.0

# 训练期实际平均 RankIC，用于观察样本外衰减。
TRAIN_MEAN_RANK_IC = 0.04437629134630530

# 门槛必须在查看验证结果前冻结；失败后不利用验证集反复调参。
GATE_MIN_DIRECTIONAL_RANK_IC = 0.02
GATE_MIN_DIRECTION_RATIO = 0.55
GATE_MIN_GROUP_MONOTONICITY = 0.70
GATE_MIN_STRICT_COST_ANNUAL_RETURN = 0.00
GATE_MIN_OBSERVATIONS = 150
GATE_MIN_FILL_RATIO = 0.90

# ============================================================================
# ========================== 用户修改区结束 ==================================
# ============================================================================


FACTOR_DIRECTION = FACTOR_DIRECTION.strip().lower()
DATE_TAG = (
    VALIDATION_START_DATE.replace('-', '')
    + '_'
    + VALIDATION_END_DATE.replace('-', '')
)

default_factor_output_dir = (
    FACTOR_DATA_ROOT / f'{FACTOR_OUTPUT_PREFIX}_{DATE_TAG}'
)
default_factor_path = (
    default_factor_output_dir / f'{FACTOR_FILE_PREFIX}_{DATE_TAG}.parquet'
)
FACTOR_PATH = Path(os.getenv(
    'VALIDATIONTEST_FACTOR_PATH', str(default_factor_path)
))
COVERAGE_PATH = Path(os.getenv(
    'VALIDATIONTEST_COVERAGE_PATH',
    str(default_factor_output_dir / 'factor_coverage.csv'),
))

default_market_path = LEARNING_DIR / 'Data' / 'stocks_daily_04-25.parquet'
MARKET_PARQUET = Path(os.getenv(
    'VALIDATIONTEST_MARKET_PARQUET', str(default_market_path)
))
FLAGS_DIR = Path(os.getenv(
    'VALIDATIONTEST_FLAGS_DIR',
    str(VALIDATION_DIR / '.cache' / 'strict_tradability_flags' / DATE_TAG),
))
CACHE_DIR = Path(os.getenv(
    'VALIDATIONTEST_CACHE_DIR',
    str(BACKTEST_DIR / '.cache' / 'market_data'),
))
default_output_dir = (
    VALIDATION_DIR / 'outputs' / FACTOR_NAME
    / f'{HORIZON}d_validation_{DATE_TAG}'
)
OUTPUT_DIR = Path(os.getenv(
    'VALIDATIONTEST_OUTPUT_DIR', str(default_output_dir)
))

TRAIN_END_TS = pd.Timestamp(TRAIN_END_DATE)
VALIDATION_START_TS = pd.Timestamp(VALIDATION_START_DATE)
VALIDATION_END_TS = pd.Timestamp(VALIDATION_END_DATE)
TEST_START_TS = pd.Timestamp(TEST_START_DATE)

assert TRAIN_END_TS < VALIDATION_START_TS <= VALIDATION_END_TS < TEST_START_TS
assert FACTOR_DIRECTION in {'positive', 'negative'}
assert EXECUTION_LAG >= 1
assert HORIZON >= 1 and REBALANCE_STEP >= HORIZON
assert STRICT_COST_GATE_BPS in COST_STRESS_BPS
assert FACTOR_PATH.is_file(), (
    f'找不到验证因子：{FACTOR_PATH}\n'
    '请先用 Factor/TURNOVER_STABILITY.ipynb 生成对应日期的因子文件。'
)
assert MARKET_PARQUET.is_file(), f'找不到行情文件：{MARKET_PARQUET}'

FLAGS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

protocol = pd.Series({
    '回测引擎': BACKTEST_ENGINE,
    '训练集截止': TRAIN_END_DATE,
    '验证集': f'{VALIDATION_START_DATE} 至 {VALIDATION_END_DATE}',
    '测试集锁定起点': TEST_START_DATE,
    '因子': f'{FACTOR_NAME} / {FACTOR_COLUMN} / {FACTOR_DIRECTION}',
    '因子文件': FACTOR_PATH,
    '行情文件': MARKET_PARQUET,
    '信号/成交/持有': f'T日收盘 / T+{EXECUTION_LAG} VWAP / {HORIZON}交易日',
    '调仓间隔': REBALANCE_STEP,
    '基准成本bps': BASE_COST_BPS,
    '严格成本门槛bps': STRICT_COST_GATE_BPS,
})
display(protocol.to_frame('冻结值'))


## 2. 读取预生成验证因子

验证因子必须与训练因子具有相同公式、字段和预处理顺序，仅日期不同。本节检查字段、日期边界、`asof_date`、重复键、数值有效性以及测试集隔离。


In [ ]:
def load_parquet_window(path, columns, start_ts, end_ts, batch_size=300_000):
    start_int = int(pd.Timestamp(start_ts).strftime('%Y%m%d'))
    end_int = int(pd.Timestamp(end_ts).strftime('%Y%m%d'))
    parquet_file = pq.ParquetFile(path)
    selected = []
    scanned = 0
    for batch in parquet_file.iter_batches(
        batch_size=batch_size, columns=columns, use_threads=True
    ):
        table = pa.Table.from_batches([batch])
        date_column = table['交易日']
        mask = pc.and_(
            pc.greater_equal(date_column, pa.scalar(start_int)),
            pc.less_equal(date_column, pa.scalar(end_int)),
        )
        filtered = table.filter(mask)
        if filtered.num_rows:
            selected.append(filtered)
        scanned += table.num_rows
    assert selected, '目标区间没有行情数据'
    result = pa.concat_tables(selected).to_pandas()
    print(f'扫描 {scanned:,} 行行情，目标区间保留 {len(result):,} 行')
    return result


factor_file = pq.ParquetFile(FACTOR_PATH)
required_columns = {'date', 'code', FACTOR_COLUMN, 'asof_date'}
missing_columns = required_columns - set(factor_file.schema_arrow.names)
assert not missing_columns, f'验证因子缺少列：{sorted(missing_columns)}'

factor_panel = pd.read_parquet(
    FACTOR_PATH,
    columns=['date', 'code', FACTOR_COLUMN, 'asof_date'],
)
factor_panel['date'] = pd.to_datetime(factor_panel['date'], errors='coerce')
factor_panel['asof_date'] = pd.to_datetime(
    factor_panel['asof_date'], errors='coerce'
)

assert not factor_panel.empty
assert factor_panel[['date', 'asof_date']].notna().all().all()
assert factor_panel['date'].min() == VALIDATION_START_TS, (
    '验证因子首日与配置不一致'
)
assert factor_panel['date'].max() == VALIDATION_END_TS, (
    '验证因子末日与配置不一致'
)
assert factor_panel['date'].max() < TEST_START_TS
assert (factor_panel['asof_date'] == factor_panel['date']).all()
assert not factor_panel.duplicated(['date', 'code']).any()
assert np.isfinite(factor_panel[FACTOR_COLUMN]).all()

factor_summary = pd.Series({
    '验证因子行数': len(factor_panel),
    '股票数量': factor_panel['code'].nunique(),
    '交易日数量': factor_panel['date'].nunique(),
    '因子首日': factor_panel['date'].min(),
    '因子末日': factor_panel['date'].max(),
    '测试期记录数': int((factor_panel['date'] >= TEST_START_TS).sum()),
    'Parquet大小MB': FACTOR_PATH.stat().st_size / 1024 / 1024,
})
display(factor_summary.to_frame('值'))

if COVERAGE_PATH.is_file():
    factor_coverage = pd.read_csv(COVERAGE_PATH)
    print('覆盖率文件：', COVERAGE_PATH)
    display(factor_coverage.tail())
else:
    print('提示：没有找到覆盖率文件，不影响回测：', COVERAGE_PATH)

del factor_panel
gc.collect()


## 3. 建立更严格的入场可交易标记

回测引擎的 `basic` 模式会检查入场日 OHLC、成交量和复权 VWAP。本节额外排除：

- 入场日名称含 ST；
- 成交量、成交额、复权因子或 OHLC 无效；
- 入场日开盘价触及涨停价或跌停价。

分组在读取未来收益前完成。不可成交股票保持目标权重但实现收益记为现金 0，不会事后换成另一只股票。

> 仍然没有逐股融券可得性和冲击成本数据，因此多空组合是研究组合；实盘判断还应重点观察多头组表现。


In [ ]:
FLAG_COLUMNS = [
    '股票代码', '股票名称', '交易日',
    '开盘价', '最高价', '最低价', '收盘价',
    '成交量（手）', '成交额（千元）', '复权因子',
    '当日涨停价', '当日跌停价',
]
flag_market = load_parquet_window(
    MARKET_PARQUET, FLAG_COLUMNS, VALIDATION_START_TS, VALIDATION_END_TS
)
flag_market = flag_market.rename(columns={
    '股票代码': 'code', '股票名称': 'name', '交易日': 'date_int',
    '开盘价': 'open', '最高价': 'high', '最低价': 'low', '收盘价': 'close',
    '成交量（手）': 'volume', '成交额（千元）': 'amount',
    '复权因子': 'adj_factor', '当日涨停价': 'limit_up',
    '当日跌停价': 'limit_down',
})
flag_market['date'] = pd.to_datetime(
    flag_market.pop('date_int'), format='%Y%m%d'
)
numeric_columns = [
    'open', 'high', 'low', 'close', 'volume', 'amount',
    'adj_factor', 'limit_up', 'limit_down',
]
for column in numeric_columns:
    flag_market[column] = pd.to_numeric(flag_market[column], errors='coerce')

valid_market = (
    flag_market['code'].notna()
    & (flag_market[['open', 'high', 'low', 'close']] > 0).all(axis=1)
    & (flag_market['volume'] > 0)
    & (flag_market['amount'] > 0)
    & (flag_market['adj_factor'] > 0)
)
entry_is_st = flag_market['name'].fillna('').str.upper().str.contains(
    'ST', regex=False
)
open_at_limit_up = (
    (flag_market['limit_up'] > 0)
    & (flag_market['open'] >= flag_market['limit_up'] - PRICE_LIMIT_TOLERANCE)
)
open_at_limit_down = (
    (flag_market['limit_down'] > 0)
    & (flag_market['open'] <= flag_market['limit_down'] + PRICE_LIMIT_TOLERANCE)
)
flag_market['tradable'] = (
    valid_market & ~entry_is_st & ~open_at_limit_up & ~open_at_limit_down
)

for date, daily in flag_market.groupby('date', sort=True):
    daily[['code', 'tradable']].to_csv(
        FLAGS_DIR / f'{date:%Y-%m-%d}.csv', index=False
    )

flag_audit = flag_market.groupby('date').agg(
    market_rows=('code', 'size'),
    tradable_rows=('tradable', 'sum'),
)
flag_audit['tradable_ratio'] = (
    flag_audit['tradable_rows'] / flag_audit['market_rows']
)
flag_reason_summary = pd.Series({
    '总记录数': len(flag_market),
    '基础行情不可交易': int((~valid_market).sum()),
    '入场日ST': int(entry_is_st.sum()),
    '开盘触及涨停': int(open_at_limit_up.sum()),
    '开盘触及跌停': int(open_at_limit_down.sum()),
    '最终可交易': int(flag_market['tradable'].sum()),
})
display(flag_reason_summary.to_frame('数量'))
display(flag_audit.describe().T)

flags_manifest = {
    'validation_start': VALIDATION_START_DATE,
    'validation_end': VALIDATION_END_DATE,
    'test_start_locked': TEST_START_DATE,
    'rule': 'valid OHLC/volume/amount/adj_factor; non-ST; open not at price limits',
    'price_limit_tolerance': PRICE_LIMIT_TOLERANCE,
    'files': int(flag_market['date'].nunique()),
}
(FLAGS_DIR / 'manifest.json').write_text(
    json.dumps(flags_manifest, ensure_ascii=False, indent=2), encoding='utf-8'
)
del flag_market
gc.collect()


## 4. 输入契约与测试集隔离检查

先确认因子文件、交易标记和日期边界正确，再运行任何未来收益计算。

In [ ]:
flag_files = sorted(FLAGS_DIR.glob('????-??-??.csv'))
assert flag_files, '没有严格交易标记文件'
assert pd.Timestamp(flag_files[0].stem) == VALIDATION_START_TS
assert pd.Timestamp(flag_files[-1].stem) == VALIDATION_END_TS
assert pd.Timestamp(flag_files[-1].stem) < TEST_START_TS

display(pd.Series({
    '因子Parquet行数': factor_file.metadata.num_rows,
    '因子首日': factor_summary['因子首日'],
    '因子末日': factor_summary['因子末日'],
    '交易标记文件数': len(flag_files),
    '交易标记首日': flag_files[0].stem,
    '交易标记末日': flag_files[-1].stem,
    '测试期数据进入验证': False,
}).to_frame('值'))


## 5. 运行严格验证回测

时间契约：T 日收盘得到因子，至少 T+1 按复权 VWAP 建仓，持有 `HORIZON` 个交易日，并按 `REBALANCE_STEP` 调仓；二者与训练回测保持一致。

验证主结果使用训练时的基础成本，后面额外执行 `COST_STRESS_BPS` 成本压力测试。缺失收益不事后剔除，目标权重保持现金收益 0；任一信号日缺失比例超过 `MAX_MISSING_RETURN_RATIO` 会直接报错。


In [ ]:
paths = DataPaths(
    factor_dir=FACTOR_PATH,
    market_parquet=MARKET_PARQUET,
    market_cache_dir=CACHE_DIR,
    output_dir=OUTPUT_DIR,
    flags_dir=FLAGS_DIR,
)
factor = FactorSpec(
    name=FACTOR_NAME,
    column=FACTOR_COLUMN,
    direction=FACTOR_DIRECTION,
    preprocessed=True,
)
config = BacktestConfig(
    start_date=VALIDATION_START_DATE,
    end_date=VALIDATION_END_DATE,
    horizon=HORIZON,
    execution_lag=EXECUTION_LAG,
    rebalance_step=REBALANCE_STEP,
    groups=GROUPS,
    cost_bps=BASE_COST_BPS,
    min_stocks=MIN_STOCKS,
    tradability_mode='basic',
    missing_return_policy='zero',
    max_missing_return_ratio=MAX_MISSING_RETURN_RATIO,
    neutralize=False,
    market_exchanges=('SH', 'SZ'),
    save_holdings=SAVE_HOLDINGS,
    allow_exit_after_end=False,
)
result = StrictFactorBacktester(paths, factor, config).run(save=True)
print('验证回测完成')
print('实际信号区间：', result.group_returns.index.min(), '至', result.group_returns.index.max())
print('有效观察期：', len(result.group_returns))
print('结果目录：', OUTPUT_DIR)

## 6. 完整性审计：先看能否相信，再看赚不赚钱

In [ ]:
audit = result.audit.copy()
assert (audit['entry_date'] > audit.index).all(), '存在同日或提前成交'
assert (audit['exit_date'] > audit['entry_date']).all(), '退出日期错误'
assert audit['exit_date'].max() <= VALIDATION_END_TS < TEST_START_TS
assert audit['missing_return_ratio'].max() <= MAX_MISSING_RETURN_RATIO
audit_summary = audit[[
    'factor_rows_raw', 'factor_rows_valid', 'fill_ratio', 'missing_return_ratio'
]].describe().T
display(audit_summary)
display(audit.nsmallest(5, 'fill_ratio'))
display(audit.nlargest(5, 'missing_return_ratio'))

## 7. RankIC、年度稳定性与训练衰减

负向因子的 RankIC 应为负。验证期不要求复制训练期收益，但方向、年度稳定性和经济意义不能消失。

In [ ]:
ic_series = result.ic_series.copy()
ic_summary = result.ic_summary.copy()
yearly_ic = ic_series.groupby(ic_series.index.year).agg(
    rank_ic_mean=('rank_ic', 'mean'),
    rank_ic_std=('rank_ic', 'std'),
    negative_ratio=('rank_ic', lambda x: (x < 0).mean()),
    observations=('rank_ic', 'count'),
    mean_stocks=('n_ic', 'mean'),
)
yearly_ic.index.name = 'year'
yearly_ic['rank_ic_ir'] = (
    yearly_ic['rank_ic_mean']
    / yearly_ic['rank_ic_std'].where(yearly_ic['rank_ic_std'] > 0)
    * np.sqrt(252 / REBALANCE_STEP)
)
validation_mean_rank_ic = float(ic_series['rank_ic'].mean())
rank_ic_retention = (
    abs(validation_mean_rank_ic) / abs(TRAIN_MEAN_RANK_IC)
)
display(ic_summary)
display(yearly_ic)
print(f'训练平均RankIC：{TRAIN_MEAN_RANK_IC:.6f}')
print(f'验证平均RankIC：{validation_mean_rank_ic:.6f}')
print(f'绝对RankIC保留比例：{rank_ic_retention:.2%}')


## 8. 分组收益与基础绩效

`period_mean` 是每个5日持有期的平均收益，不是单日收益。年化均值使用 `252 / 5` 个非重叠周期。

In [ ]:
group_returns = result.group_returns.copy()
group_columns = [f'Q{i}' for i in range(1, GROUPS + 1)]
periods_per_year = 252 / REBALANCE_STEP
group_summary = pd.DataFrame({
    'period_mean': group_returns[group_columns].mean(),
    'annualized_mean': group_returns[group_columns].mean() * periods_per_year,
})
performance = result.performance.copy()
display(group_summary)
display(performance)
display(group_returns[[
    'long_turnover', 'short_turnover', 'transaction_cost',
    'long_short_gross', 'long_short_net'
]].describe().T)

## 9. 交易成本压力测试

组合和持仓保持完全不变，只改变双边换手对应的成本。`STRICT_COST_GATE_BPS` 对应结果用于正式通过门槛。


In [ ]:
def return_metrics(returns, periods_per_year):
    values = pd.to_numeric(returns, errors='coerce').dropna()
    nav = (1 + values).cumprod()
    annual_return = nav.iloc[-1] ** (periods_per_year / len(values)) - 1
    annual_volatility = values.std(ddof=1) * np.sqrt(periods_per_year)
    drawdown = nav / nav.cummax() - 1
    return {
        'total_return': nav.iloc[-1] - 1,
        'annual_return': annual_return,
        'annual_volatility': annual_volatility,
        'sharpe': annual_return / annual_volatility if annual_volatility > 0 else np.nan,
        'max_drawdown': drawdown.min(),
        'win_rate': (values > 0).mean(),
    }


total_turnover = group_returns['long_turnover'] + group_returns['short_turnover']
cost_stress_rows = []
for cost_bps in COST_STRESS_BPS:
    stressed_return = (
        group_returns['long_short_gross'] - total_turnover * cost_bps / 10_000
    )
    cost_stress_rows.append({
        'cost_bps': cost_bps,
        **return_metrics(stressed_return, periods_per_year),
    })
cost_stress = pd.DataFrame(cost_stress_rows).set_index('cost_bps')
display(cost_stress)
cost_stress.to_csv(OUTPUT_DIR / 'cost_stress.csv')


## 10. 按预先写死的门槛给出验证结论

门槛在读取验证结果之前已经写入第1节。没有通过时应记录失败，而不是回头改参数直到通过。

In [ ]:
directional_rank_ic = (
    -validation_mean_rank_ic
    if FACTOR_DIRECTION == 'negative' else validation_mean_rank_ic
)
direction_ratio = float(
    (ic_series['rank_ic'] < 0).mean()
    if FACTOR_DIRECTION == 'negative'
    else (ic_series['rank_ic'] > 0).mean()
)
group_number = pd.Series(range(1, GROUPS + 1), index=group_columns, dtype=float)
raw_group_spearman = group_number.corr(
    group_summary['annualized_mean'], method='spearman'
)
group_monotonicity = (
    -raw_group_spearman
    if FACTOR_DIRECTION == 'negative' else raw_group_spearman
)
strict_cost_annual_return = float(
    cost_stress.loc[STRICT_COST_GATE_BPS, 'annual_return']
)
minimum_fill_ratio = float(audit['fill_ratio'].min())
maximum_missing_ratio = float(audit['missing_return_ratio'].max())

validation_gates = pd.DataFrame([
    {
        '检查项': '方向调整后平均RankIC', '实际值': directional_rank_ic,
        '门槛': f'>= {GATE_MIN_DIRECTIONAL_RANK_IC:.2f}',
        '通过': directional_rank_ic >= GATE_MIN_DIRECTIONAL_RANK_IC,
    },
    {
        '检查项': 'RankIC方向命中率', '实际值': direction_ratio,
        '门槛': f'>= {GATE_MIN_DIRECTION_RATIO:.0%}',
        '通过': direction_ratio >= GATE_MIN_DIRECTION_RATIO,
    },
    {
        '检查项': '十分组方向单调性', '实际值': group_monotonicity,
        '门槛': f'>= {GATE_MIN_GROUP_MONOTONICITY:.2f}',
        '通过': group_monotonicity >= GATE_MIN_GROUP_MONOTONICITY,
    },
    {
        '检查项': f'{STRICT_COST_GATE_BPS:.0f}bps成本后多空年化',
        '实际值': strict_cost_annual_return,
        '门槛': f'> {GATE_MIN_STRICT_COST_ANNUAL_RETURN:.0%}',
        '通过': strict_cost_annual_return > GATE_MIN_STRICT_COST_ANNUAL_RETURN,
    },
    {
        '检查项': '有效观察期', '实际值': len(ic_series),
        '门槛': f'>= {GATE_MIN_OBSERVATIONS}',
        '通过': len(ic_series) >= GATE_MIN_OBSERVATIONS,
    },
    {
        '检查项': '最低成交覆盖率', '实际值': minimum_fill_ratio,
        '门槛': f'>= {GATE_MIN_FILL_RATIO:.0%}',
        '通过': minimum_fill_ratio >= GATE_MIN_FILL_RATIO,
    },
    {
        '检查项': '最大缺失收益比例', '实际值': maximum_missing_ratio,
        '门槛': f'<= {MAX_MISSING_RETURN_RATIO:.0%}',
        '通过': maximum_missing_ratio <= MAX_MISSING_RETURN_RATIO,
    },
])
overall_pass = bool(validation_gates['通过'].all())
verdict = (
    '验证通过：保留为候选因子，但仍禁止查看测试集'
    if overall_pass
    else '验证未通过：记录结果，不使用测试集调参'
)
display(validation_gates)
print()
print('最终结论：', verdict)
validation_gates.to_csv(OUTPUT_DIR / 'validation_gates.csv', index=False)

## 11. 图表与可复现记录

In [ ]:
os.environ.setdefault('MPLCONFIGDIR', str(VALIDATION_DIR / '.cache' / 'matplotlib'))
import matplotlib.pyplot as plt

if FACTOR_DIRECTION == 'negative':
    long_group, short_group = 'Q1', f'Q{GROUPS}'
else:
    long_group, short_group = f'Q{GROUPS}', 'Q1'
nav = (1 + group_returns[[
    long_group, short_group, 'long_short_net'
]].fillna(0)).cumprod()
rolling_rank_ic = ic_series['rank_ic'].rolling(20, min_periods=10).mean()

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
rolling_rank_ic.plot(ax=axes[0, 0], color='tab:blue', linewidth=1)
axes[0, 0].axhline(0, color='black', linewidth=0.8)
axes[0, 0].set_title('20-period Rolling RankIC')
group_summary['annualized_mean'].plot.bar(ax=axes[0, 1], color='tab:orange')
axes[0, 1].set_title('Annualized Mean Return by Group')
axes[0, 1].tick_params(axis='x', rotation=0)
nav.plot(ax=axes[1, 0], linewidth=1.2)
axes[1, 0].axhline(1, color='black', linewidth=0.8)
axes[1, 0].set_title('Validation NAV')
audit[['fill_ratio', 'missing_return_ratio']].plot(ax=axes[1, 1], linewidth=1)
axes[1, 1].set_title('Execution Integrity')
for ax in axes.flat:
    ax.grid(alpha=0.25)
    ax.set_xlabel('')
plt.tight_layout()
dashboard_path = OUTPUT_DIR / 'validation_dashboard.png'
fig.savefig(dashboard_path, dpi=160, bbox_inches='tight')
plt.show()

yearly_ic.to_csv(OUTPUT_DIR / 'yearly_ic.csv')
group_summary.to_csv(OUTPUT_DIR / 'group_summary.csv')
protocol_record = {
    'factor': FACTOR_NAME,
    'factor_column': FACTOR_COLUMN,
    'factor_direction': FACTOR_DIRECTION,
    'factor_path': str(FACTOR_PATH),
    'training_end': TRAIN_END_DATE,
    'validation_start': VALIDATION_START_DATE,
    'validation_end': VALIDATION_END_DATE,
    'test_start_locked': TEST_START_DATE,
    'horizon': HORIZON,
    'execution_lag': EXECUTION_LAG,
    'rebalance_step': REBALANCE_STEP,
    'base_cost_bps': BASE_COST_BPS,
    'cost_stress_bps': list(COST_STRESS_BPS),
    'strict_cost_gate_bps': STRICT_COST_GATE_BPS,
    'train_mean_rank_ic': TRAIN_MEAN_RANK_IC,
    'overall_pass': overall_pass,
    'verdict': verdict,
}
(OUTPUT_DIR / 'validation_protocol.json').write_text(
    json.dumps(protocol_record, ensure_ascii=False, indent=2), encoding='utf-8'
)
print('验证图表：', dashboard_path)
print('验证门槛：', OUTPUT_DIR / 'validation_gates.csv')
print('验证协议：', OUTPUT_DIR / 'validation_protocol.json')


## 12. 看完验证结果后该怎么做

- **全部通过**：把 STR 标记为候选因子；测试集从 2022-01-04 起仍保持锁定。
- **部分未通过**：记录失败项目和可能原因，但不要利用验证结果修改20日因子窗口、方向、去极值参数、持有期或验证门槛。
- **数据完整性失败**：可以修复代码或数据契约后重跑，因为这不是策略调参；必须留下修复记录。
- **最终测试**：等多个候选因子、组合方式和权重全部确定后，对2022年以后只运行一次。

严格交易标记缓存位于 `.cache/strict_tradability_flags/{DATE_TAG}`，可重新生成；正式因子、验证结论和回测结果应保留，以便记录完整研究路径。
